# Episode 1 Lab Notebook: I Built My Own AI Assistant for Free

**Series:** genai-dev-tv &nbsp;|&nbsp; **Episode:** 01 &nbsp;|&nbsp; **Exam module:** AIP-C01 Domain 1 — Foundation Model Integration

This notebook walks through everything in `01_bedrock_basics.py`, one cell at a time, so you (or your
video viewers) can run it interactively, inspect the raw API responses, and see exactly what the
Converse API returns before it gets wrapped into a script or a Streamlit app.

**What you'll do:**
1. Connect to Bedrock and discover which models you have access to.
2. Call Claude, Titan, and Nova through **one** API (`Converse`) — and see why that matters.
3. Compare response shape, latency, and token usage across model families.
4. Estimate cost per call and log a mental model for Episode 6 ("I Cut My AI Bill 90%").

> **Prerequisite:** In the Bedrock console → *Model access*, request access to at least one
> Anthropic Claude model, one Amazon Titan Text model, and the Amazon Nova family. Approval is
> usually instant for these.


In [ ]:
import boto3, json
from botocore.exceptions import ClientError

REGION = "us-east-1"  # widest model selection; change if you deployed elsewhere
session = boto3.Session(region_name=REGION)

bedrock = session.client("bedrock")          # control plane: list/describe models
bedrock_runtime = session.client("bedrock-runtime")  # data plane: run inference

print("Connected. Identity:", boto3.client("sts").get_caller_identity()["Arn"])


## 1. Discover available models

Bedrock's model catalog changes constantly — new versions ship, old ones enter Legacy/EOL status.
Rather than hard-code a model ID that might be retired by the time you watch this, we ask the
`bedrock` control-plane API what's actually enabled in *your* account right now.

This is directly testable on the exam: knowing the difference between the **`bedrock`** (control
plane — `ListFoundationModels`, `GetFoundationModel`, custom model + provisioned throughput
management) and **`bedrock-runtime`** (data plane — `InvokeModel`, `Converse`, `CountTokens`)
endpoints is a Domain 1 fundamental.


In [ ]:
resp = bedrock.list_foundation_models(byOutputModality="TEXT")
all_models = resp["modelSummaries"]

families = ["anthropic.claude", "amazon.titan-text", "amazon.nova"]
for family in families:
    matches = [m["modelId"] for m in all_models if m["modelId"].startswith(family)]
    print(f"{family:22s} -> {len(matches)} model(s), e.g. {matches[:2]}")


## 2. One API, three model families: the Converse API

`InvokeModel` needs a **different JSON request body per model provider** (Anthropic's shape isn't
Titan's shape isn't Nova's shape). `Converse` normalizes all of that into one request and one
response schema: `messages`, `system`, `inferenceConfig`, and optionally `toolConfig` /
`guardrailConfig` for later episodes.

That's the whole reason this lab — and the chat app you'll build next — can offer a **model
selector dropdown** without an if/elif chain per provider.


In [ ]:
def ask(model_id: str, prompt: str) -> dict:
    response = bedrock_runtime.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        system=[{"text": "Answer in 2-3 concise sentences for a YouTube tutorial audience."}],
        inferenceConfig={"maxTokens": 300, "temperature": 0.5, "topP": 0.9},
    )
    content = response["output"]["message"]["content"]
    return {
        "model_id": model_id,
        "reply": "".join(b.get("text", "") for b in content),
        "usage": response["usage"],
        "latency_ms": response["metrics"]["latencyMs"],
    }

PROMPT = "Explain what a foundation model is, for someone who has never used AI before."

# Pick one on-demand model per family that's actually enabled in your account.
chosen = {}
for family in families:
    candidates = [
        m["modelId"] for m in all_models
        if m["modelId"].startswith(family)
        and "ON_DEMAND" in m.get("inferenceTypesSupported", [])
    ]
    if candidates:
        chosen[family] = sorted(candidates)[-1]

chosen


In [ ]:
results = {}
for family, model_id in chosen.items():
    try:
        results[family] = ask(model_id, PROMPT)
    except ClientError as e:
        print(f"{family}: request access to {model_id} in the Bedrock console -> {e}")

for family, r in results.items():
    print(f"\n=== {family} ({r['model_id']}) ===")
    print(r["reply"])
    print(f"tokens: {r['usage']['inputTokens']} in / {r['usage']['outputTokens']} out  |  {r['latency_ms']}ms")


## 3. Compare the three responses side by side

Same prompt, same request shape, three different models. Notice the differences in tone, length,
latency, and token usage — this is the raw material for Episode 6's cost/quality tradeoff
discussion, and for the chat app's "pick cheap vs. smart per task" design.


In [ ]:
import pandas as pd

rows = []
for family, r in results.items():
    rows.append({
        "family": family,
        "model_id": r["model_id"],
        "input_tokens": r["usage"]["inputTokens"],
        "output_tokens": r["usage"]["outputTokens"],
        "latency_ms": r["latency_ms"],
        "reply_preview": r["reply"][:80] + ("..." if len(r["reply"]) > 80 else ""),
    })

df = pd.DataFrame(rows)
df


## 4. Estimate cost per call

Bedrock bills per 1,000 input/output tokens, and the rate differs a lot by model size. The table
below uses **illustrative** prices (check the [Bedrock pricing page](https://aws.amazon.com/bedrock/pricing/)
for current numbers) just to make the *relative* cost gap visible — this is the number that makes
"$0/month when idle, fractions of a cent per message" true in the video.


In [ ]:
PRICE_PER_1K = {
    "anthropic.claude": {"input": 0.003, "output": 0.015},
    "amazon.titan-text": {"input": 0.0002, "output": 0.0006},
    "amazon.nova": {"input": 0.00006, "output": 0.00024},  # Nova Lite-ish, illustrative
}

for family, r in results.items():
    price = PRICE_PER_1K.get(family, {"input": 0, "output": 0})
    cost = (r["usage"]["inputTokens"] / 1000) * price["input"] + (r["usage"]["outputTokens"] / 1000) * price["output"]
    print(f"{family:22s} ${cost:.6f} for this one call")


## 5. What's next

- Run `01_chat_app.py` (Streamlit) to put this behind a real chat UI with a model selector.
- Run `01_cost_tracker.py` to see the **CountTokens** API estimate cost *before* you spend it,
  and push the actual usage to CloudWatch as a custom metric.
- Deploy `cdk/` to put this behind API Gateway + Lambda + S3 chat history — the exact
  architecture in `architecture.mmd`.
- See `EXAM_NOTES.md` for the AIP-C01 Domain 1 study notes this lab maps to.
